# Лабораторная работа № 4. Вариант 6 Романов Михаил ПМИ-3

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
HERE = Path(".")

Подключены библиотеки и настроен вывод таблиц.

## 1. Диагностика источника

Протокол: ответ сервера через requests, поиск карточек в HTML, наличие JSON в `<script>`, XHR-эндпоинт из обработчика кнопки.

In [2]:
import re
import requests
from bs4 import BeautifulSoup

UA = "StudentScraperBot/1.0 (+https://github.com/PINNOS/parsing)"
r = requests.get("https://www.scrapingcourse.com/button-click", headers={"User-Agent": UA}, timeout=15)
soup = BeautifulSoup(r.text, "html.parser")
print("HTTP:", r.status_code, "| размер:", len(r.text), "байт")
print("карточек .product-name в статике:", len(soup.select(".product-name")))
print("кнопка Load more:", [b.get_text(strip=True) for b in soup.select("button")])
print("__NEXT_DATA__:", "__NEXT_DATA__" in r.text, "| ld+json:", len(soup.select('script[type="application/ld+json"]')))
m = re.search(r"fetch\(`([^`]+)`", r.text)
print("XHR из обработчика кнопки:", m.group(1) if m else None)

HTTP: 200 | размер: 15556 байт
карточек .product-name в статике: 12
кнопка Load more: ['Load more']
__NEXT_DATA__: False | ld+json: 0
XHR из обработчика кнопки: /ajax/products?offset=${offset}


В HTML есть только первые 12 товаров. Кнопка загружает остальные через /ajax/products, поэтому данные можно получить и без браузера

## 2. Браузерный сбор (Playwright)

Два раздела. цикл Load more (клики кнопки, ожидание ответа и роста DOM) и пагинация next на books.toscrape.com. Предохранители меньше равно30 циклов, стоп после 2 циклов без прироста, лимит записей

In [3]:
full = json.loads((HERE / "run_summary_full.json").read_text(encoding="utf-8"))
print({k: full[k] for k in ("records_unique", "records_raw", "by_source", "cycles", "elapsed_s", "exit_code")})
log = pd.read_csv(HERE / "collection_log.csv", encoding="utf-8-sig")
log

{'records_unique': 227, 'records_raw': 255, 'by_source': {'load_more': 147, 'books': 80}, 'cycles': 19, 'elapsed_s': 28.49, 'exit_code': 0}


,source,cycle,elements,delta,duration_ms,timestamp,url,http_status
0,load_more,1,24,12,278,2026-10-17T14:49:48+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
1,load_more,2,36,12,270,2026-10-17T14:49:49+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
2,load_more,3,48,12,216,2026-10-17T14:49:51+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
3,load_more,4,60,12,323,2026-10-17T14:49:52+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
4,load_more,5,72,12,239,2026-10-17T14:49:53+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
5,load_more,6,84,12,352,2026-10-17T14:49:54+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
6,load_more,7,96,12,447,2026-10-17T14:49:56+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
7,load_more,8,108,12,366,2026-10-17T14:49:57+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
8,load_more,9,120,12,247,2026-10-17T14:49:59+00:00,https://www.scrapingcourse.com/ajax/products?o...,200
9,load_more,10,132,12,241,2026-10-17T14:50:00+00:00,https://www.scrapingcourse.com/ajax/products?o...,200


Собрано 227 уникальных записей: 147 товаров и 80 книг. Из 255 сырых записей 28 оказались повторами из-за перекрытия порций. Последний ответ вернул статус 204, после чего загрузка закончилась

## 3.requests и XHR-эндпоинт

Тот же набор без браузера: `GET /ajax/products?offset=N` с шагом 10 и статическая пагинация books. Сверка с браузерным набором по ключу

In [4]:
light = json.loads((HERE / "run_summary_light.json").read_text(encoding="utf-8"))
print({k: light[k] for k in ("records_unique", "requests", "retries", "elapsed_s", "exit_code")})
print(json.dumps(light["reconcile"], ensure_ascii=False, indent=1))

{'records_unique': 227, 'requests': 20, 'retries': 0, 'elapsed_s': 25.17, 'exit_code': 0}
{
 "matched_by_business_key": 227,
 "only_browser": 0,
 "only_light": 0,
 "name_mismatch": 0,
 "price_mismatch_numeric": 0,
 "url_mismatch": 0
}


 Оба способа дали одинаковые 227 записей. Названия, цены и ссылки совпали. При сверке исправлены кодировка книг и очистка цен.

## 4. Замеры и сравнение подходов

Метрики берутся из run_summary_*.json. Время прогона в основном определяется паузой 1 с между циклами

In [5]:
rows = []
for name, file in [("Playwright (полный)", "run_summary_full.json"),
                   ("Playwright (--block-media)", "run_summary_blocked.json"),
                   ("requests + XHR", "run_summary_light.json")]:
    s = json.loads((HERE / file).read_text(encoding="utf-8"))
    rows.append({"подход": name, "записей": s["records_unique"], "время, с": s["elapsed_s"],
                 "с/запись": s["sec_per_record"], "ответов/запросов": s.get("network_responses", s.get("requests")),
                 "трафик, МБ": s["traffic_mb"], "память, МБ": s["peak_rss_mb"]})
metrics = pd.DataFrame(rows)
metrics

,подход,записей,"время, с",с/запись,ответов/запросов,"трафик, МБ","память, МБ"
0,Playwright (полный),227,28.49,0.1255,322,13.741,977.0
1,Playwright (--block-media),227,29.38,0.1294,69,0.030,440.5
2,requests + XHR,227,25.17,0.1109,20,0.338,44.2


 Блокировка медиа снизила трафик с 13,74 до 0,03 МБ и расход памяти с 977 до 441 МБ. Время почти не изменилось из-за пауз между запросами. Вариант с requests использовал 0,34 МБ трафика и 44 МБ памяти

## 5. Качество данных



In [6]:
quality = json.loads((HERE / "quality_report.json").read_text(encoding="utf-8"))
pd.DataFrame([{"набор": q["dataset"], "записей": q["records"], "вердикт": q["verdict"],
               "дубли по ключу": q["duplicate_keys_by_name"], "нечисловая цена": q["price_not_numeric"],
               "пусто name": q["fields"]["name"]["empty_share"], "пусто price": q["fields"]["price"]["empty_share"],
               "пусто url": q["fields"]["url"]["empty_share"]} for q in quality])

,набор,записей,вердикт,дубли по ключу,нечисловая цена,пусто name,пусто price,пусто url
0,data_browser.csv,227,green,0,0,0.0,0.0,0.0
1,data_light.csv,227,green,0,0,0.0,0.0,0.0


 В обоих наборах нет пустых обязательных полей, дублей и нечисловых цен. Проверка не обнаружит неверное, но непустое значение, Поэтому полезно сравнивать данные с другим способом сбора.

## 6. No-code: n8n

In [7]:
n8n = pd.read_csv(HERE / "nocode" / "nocode_result.csv", encoding="utf-8-sig")
browser_lm = pd.read_csv(HERE / "data_browser.csv", encoding="utf-8-sig")
browser_lm = browser_lm[browser_lm["source"] == "load_more"]
key = lambda s: s.str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
kn, kb = set(key(n8n["name"])), set(key(browser_lm["name"]))
print("строк в n8n:", len(n8n), "| строк в браузере (load_more):", len(browser_lm))
print("только в n8n:", len(kn - kb), "| только в браузере:", len(kb - kn))
print("пустых name/price:", int(n8n["name"].isna().sum()), int(n8n["price"].isna().sum()))

строк в n8n: 147 | строк в браузере (load_more): 147
только в n8n: 0 | только в браузере: 0
пустых name/price: 0 0


 n8n собрал 147 записей. Они совпали с браузерным набором по ключу. До удаления дублей было 175 карточек. Узел Remove Duplicates не справился, поэтому дубли удалены в Code-узле. Сценарий собирает только товары из Load more, книги в него не входят. Ветка ошибки есть, но искусственный сбой не проверял. Трафик и память n8n не измерялись